# Interferometric Image Reconstruction with OITOOLS.jl
### New Visions in Optical Interferometry Workshop — March 2026

**Presenter:** Ryan Norris  
**Duration:** 90 minutes

---

**Learning objectives.** By the end of this session you will be able to:

1. Load and critically inspect OIFITS data before attempting any reconstruction
2. Understand the ill-posed inverse problem and why regularization is unavoidable
3. Run and tune reconstructions with `reconstruct` (gradient descent / TV)
4. Run `reconstruct_bsmem` (Maximum Entropy) and compare results
5. Apply the three mandatory reliability checks before believing any feature
6. Adapt the full workflow to your own OIFITS data

**Structure**

| Part | Topic | Est. time |
|------|-------|-----------|
| 0 | Environment setup | 5 min |
| 1 | 2004 benchmark — ground truth known | 20 min |
| 2 | BSMEM on the benchmark | 10 min |
| 3 | λ Andromedae — real CHARA/MIRC data | 35 min |
| 4 | Validation: can you trust the image? | 10 min |
| 5 | Bring your own data template | 10 min |

> **Tip:** Cells marked `# ── YOUR TURN ──` are interactive exercises. Everything else runs as-is.


---
## Part 0 — Environment Setup

 **Run this cell first and wait for it to complete before anything else.**  
On a fresh Binder instance this takes ~2–3 minutes (precompilation). Subsequent runs are ~10 seconds.

In [ ]:
using OITOOLS
using PyPlot, PyCall
using Statistics
# set_oiplot_defaults() — disabled: matplotlib 3.13 compatibility issue on Binder
println("OITOOLS loaded. Julia version: ", VERSION)
# Patch set_oiplot_defaults for matplotlib 3.10+ compatibility
# The original passes font.size as an array; this fixes it.
@eval OITOOLS function set_oiplot_defaults(; compact=false)
    rcParams = PyCall.PyDict(PyCall.pyimport("matplotlib")."rcParams")
    sz = compact ? 11 : 14
    rcParams["font.size"] = sz
    rcParams["axes.titlesize"] = sz
    rcParams["axes.labelsize"] = sz
end



## All data files are included in the workshop repository — no download needed.

In [ ]:

mkpath("data")
for fname in ["2004-data1.oifits", "2004true.fits", "2011Sep10_lam_And_prepped.oifits"]
    if isfile(joinpath("data", fname))
        println("  $fname present.")
    else
        println("  WARNING: $fname not found in data/")
    end
end
println("\nAll data files ready.")

---
## Part 1 — The 2004 Imaging Beauty Contest Benchmark

We start with a dataset where **the true image is known**. The 2004 contest asked different groups to reconstruct an image from synthetic interferometric data; we now know the ground truth. This lets us see exactly what different algorithmic choices do — including where artifacts come from.

### 1.1 — Ground truth

This is what a perfect reconstruction would look like.


In [ ]:
fitsfile     = "./data/2004true.fits"
pixsize_true = 0.101   # mas/pixel
x_true       = readfits(fitsfile)
nx_true      = Int(sqrt(length(x_true)))

imdisp(x_true; pixsize=pixsize_true, tickinterval=1.0, use_colorbar=true)
println("Ground truth: $(nx_true)×$(nx_true) pixels, $(pixsize_true) mas/pixel")
println("Field of view: $(round(nx_true * pixsize_true, digits=1)) mas")


### 1.2 — Load and inspect the data

**Always look at the data before reconstructing.** The UV coverage, V² curve, and closure phases each tell you something different about what the reconstruction can and cannot recover.


In [ ]:
oifitsfile = "./data/2004-data1.oifits"
data = readoifits(oifitsfile)[1,1]   # [epoch index, wavelength index]

println("Data summary")
println("  Baseline points:   ", data.nuv)
println("  V² measurements:   ", data.nv2)
println("  T3 amplitudes:     ", data.nt3amp)
println("  Closure phases:    ", data.nt3phi)
println("  Max baseline:      ", round(maximum(data.uv_baseline), digits=1), " m")
println("  Wavelength:        ", round(mean(data.uv_lam)*1e6, digits=2), " μm")
res_mas = mean(data.uv_lam) / (2*maximum(data.uv_baseline)) * 206265e3
println("  Resolution λ/2B:   ", round(res_mas, digits=3), " mas")


In [ ]:
# UV coverage — the set of spatial frequencies we have sampled
uvplot(data)


In [ ]:
# V² vs baseline — first null gives approximate object size
plot_v2(data; logplot=true, legend_below=true)


In [ ]:
# Closure phases — non-zero values signal brightness asymmetry
plot_t3phi(data)
println("Are the closure phases consistent with zero?")
println("If yes → the source is centrosymmetric (ring, disk).")
println("If no  → there is a brightness asymmetry (companion, spot, off-center emission).")


### 1.3 — The image reconstruction problem

Interferometry measures the **Fourier transform** of the sky brightness at a discrete, incomplete set of spatial frequencies. Image reconstruction is the ill-posed inverse problem of recovering the image from this incomplete sampling.

We minimize the cost function:

$$J(\mathbf{x}) = \chi^2(\mathbf{x}) + \mu \, R(\mathbf{x})$$

- $\chi^2(\mathbf{x})$: data fidelity — how well the image matches V², T3amp, T3phi  
- $R(\mathbf{x})$: regularizer — encodes prior knowledge about what good images look like  
- $\mu$: hyperparameter — controls the trade-off between fitting data and enforcing the prior

**The fundamental tension:** a smaller $\mu$ fits the data better but can overfit noise; a larger $\mu$ is smoother but can erase real structure.

### 1.4 — Grid setup and starting image

We discretize the sky onto an $n_x \times n_x$ pixel grid with pixel scale `pixsize` (mas/pixel).

**Nyquist rule:** `pixsize` ≲ $\lambda / (2 B_{\max})$. For this dataset at H band with $B_{\max} \approx 200$ m: $\lambda/(2B_{\max}) \approx 0.21$ mas. We use 0.1 mas (oversampled by 2×, which helps gradient descent).


In [ ]:
pixsize = 0.1    # mas/pixel
nx      = 128    # → FOV = 12.8 mas

ft = setup_nfft(data, nx, pixsize)
println("Image grid: $(nx)×$(nx) pixels")
println("Pixel scale: $(pixsize) mas/pixel")
println("Field of view: $(nx*pixsize) mas")


In [ ]:
# Starting image: a broad Gaussian, normalized to unit flux.
# We don't want to start from the true image — that would be cheating.
x_start = gaussian2d(nx, nx, nx/6)
x_start = x_start / sum(x_start)

imdisp(x_start; pixsize=pixsize)
println("Starting image FWHM: $(round(nx/6 * pixsize, digits=2)) mas")


In [ ]:
# How poorly does the starting image fit the data?
chi2_f(x_start, ft, data; verb=true)
println("")
println("chi²/N >> 1 is expected — the Gaussian is just a starting point.")


### 1.5 — Regularizer catalog

| Regularizer | Effect | Best for |
|---|---|---|
| `centering` | Penalizes centroid offset from image center | Always include |
| `tv` | Edge-preserving smoothness ($\sum \|\nabla x\|_1$) | Spots, sharp disks, companions |
| `tvsq` | Smooth edges ($\sum \|\nabla x\|_2^2$) | Diffuse emission |
| `entropy` | Concentrates flux, suppresses empty regions | Smooth dust shells |
| `compactness` | Penalizes flux × distance² from center | Rapid rotators |
| `l1l2` | Intermediate between sparse and smooth | General purpose |
| `support` | Hard/soft pixel mask | When disk geometry is known |

You can combine multiple regularizers — their gradients simply add. The weight $\mu$ must be chosen empirically (see the L-curve, Part 3).

### 1.6 — First reconstruction


In [ ]:
regularizers = [
    ["centering", 1e3],
    ["l1l2",      7e6, 1e-3],
]

println("Running reconstruction (200 iterations)...")
x = reconstruct(x_start, data, ft;
                regularizers=regularizers, verb=true, maxiter=200)
println("\nDone.")


In [ ]:
# recenter() shifts the image so the brightest peak is at the center.
# The mask clips faint noise outside the main source region.
imdisp(recenter(x; mask = x .> maximum(x)/10); pixsize=pixsize, use_colorbar=true)


In [ ]:
chi2 = chi2_f(x, ft, data; verb=true)
println("")
println("Interpretation:")
println("  chi²/N ≈ 1.0  → fitting the data to within the noise (ideal)")
println("  chi²/N >> 1   → underfitting (μ too large, over-smoothed)")
println("  chi²/N << 1   → overfitting (μ too small, fitting noise)")


In [ ]:
# Always compare model observables to data.
# An image can look beautiful and still fit the data poorly.
v2_model, t3phi_model, t3amp_model = image_to_obs(x, ft, data)
plot_v2_residuals(data, v2_model)
plot_t3phi_residuals(data, t3phi_model)


### 1.7 — Regularizer comparison

Here we reconstruct the same data with four regularizers, using individually tuned $\mu$ values. Because the ground truth is available, we can score each result directly.

**Key question:** a lower chi²/N does not necessarily mean a better image. Why not?


In [ ]:
reg_variants = [
    ("tv",          [["centering", 1e3], ["tv",          1e2]]),
    ("tvsq",        [["centering", 1e3], ["tvsq",        5e7]]),
    ("entropy",     [["centering", 1e3], ["entropy",     7e2]]),
    ("compactness", [["centering", 1e3], ["compactness", 7e5, 20.0]]),
]

results_2004 = Dict{String, Matrix{Float64}}()
for (name, regs) in reg_variants
    print("Reconstructing: $name ... ")
    xr = reconstruct(x_start, data, ft;
                     regularizers=regs, verb=false, maxiter=300)
    results_2004[name] = xr
    chi2 = chi2_f(xr, ft, data)
    println("chi²/N = $(round(chi2, digits=3))")
end
println("\nAll done.")


In [ ]:
# Stack all regularizer results into a cube for side-by-side comparison
cube = zeros(nx, nx, length(reg_variants))
for (i, (name, _)) in enumerate(reg_variants)
    xr = results_2004[name]
    cube[:, :, i] = recenter(xr; mask = xr .> maximum(xr)/10)
end
labels = ["$name (χ²/N=$(round(chi2_f(results_2004[name], ft, data), digits=3)))"
          for (name, _) in reg_variants]
imdisp_multi(cube; labels, pixsize=pixsize, use_colorbar=true,
             figtitle="Regularizer comparison — 2004 benchmark")

**Discussion questions:**

1. Which regularizer best recovers the ring structure? The faint companion?
2. `entropy` tends to produce very smooth, compact images. What prior does entropy encode, and why does that make it dangerous when searching for companions?
3. `tv` can produce flat regions separated by sharp edges ("staircase" artifact). Can you see this above?
4. Does the regularizer with the **lowest** chi²/N give the **best** image? What does this tell you about using chi² alone as a quality metric?


---
## Part 2 — Maximum Entropy Reconstruction with BSMEM

OITOOLS includes a second reconstruction engine: `reconstruct_bsmem`, which implements the **BSMEM** (BiSpectrum Maximum Entropy Method) algorithm (Buscher 1994; Baron & Young 2008).

It differs from `reconstruct` in two important ways:

1. **Prior image**: BSMEM takes an explicit prior image $\mathbf{p}$ and minimises relative entropy $\sum x_i \ln(x_i / p_i)$. Structure only emerges where the data *demand* deviation from the prior.
2. **Optimizer**: BSMEM uses a different optimizer (VMLMB with bispectrum weighting) that can be more robust for certain data configurations.

Because it requires a stronger deviation from the prior to introduce any feature, BSMEM tends to be *more conservative* than gradient descent — which makes it an excellent cross-check. **A feature that appears in both TV and BSMEM is much more credible than one that appears in only one.**


In [ ]:
# Reload the 2004 benchmark (clean state)
data_bsmem = readoifits("./data/2004-data1.oifits")[1,1]
nx_b       = 128
pixsize_b  = 0.1
ft_b       = setup_nfft(data_bsmem, nx_b, pixsize_b)

# Prior: a broad Gaussian — the weakest informative prior.
# "The source is somewhere near the centre." Nothing more.
prior_fwhm = nx_b * pixsize_b / 5.0
prior      = gaussian_prior(nx_b, pixsize_b; fwhm_mas=prior_fwhm)

println("Prior FWHM: $(round(prior_fwhm, digits=2)) mas")
imdisp(prior; pixsize=pixsize_b)

In [ ]:
println("Running BSMEM reconstruction (~1–2 min)...")
x_bsmem = reconstruct_bsmem(prior, data_bsmem, ft_b;
              regularizers = [["mem", prior]],
              method       = [4, 1, 1, 2],
              maxiter      = 100,
              verbose      = true,
              flux_err     = 1e-5,
              nrand        = 10)

println("\nDone.")
imdisp(x_bsmem; pixsize=pixsize_b, use_colorbar=true)

In [ ]:
x_tv_ref = results_2004["tv"]
x_tv_c = recenter(x_tv_ref; mask = x_tv_ref .> maximum(x_tv_ref)/10)

cube_comp = zeros(nx, nx, 2)
cube_comp[:, :, 1] = x_tv_c
cube_comp[:, :, 2] = x_bsmem
c2_tv = round(chi2_f(x_tv_ref, ft, data), digits=3)
c2_bs = round(chi2_f(x_bsmem, ft_b, data_bsmem), digits=3)
imdisp_multi(cube_comp;
    labels=["TV (χ²/N=$c2_tv)", "BSMEM (χ²/N=$c2_bs)"],
    pixsize=pixsize, use_colorbar=true,
    figtitle="TV vs BSMEM — 2004 benchmark")

**Discussion:**

- Both methods should recover the ring. Does the companion appear in both?
- Notice BSMEM's image is generally smoother. Is that always a virtue?
- Which would you trust more for a publication claim of a faint companion? Why?


---
## Part 3 — Real Data: λ Andromedae (CHARA/MIRC, H band)

λ And is a bright RS CVn giant ($V = 3.8$, spectral type G8 III–IV) in a close binary system. It is rapidly rotating and magnetically active, producing **large cool starspots** visible through optical interferometry. This is real CHARA data from Martinez et al. (2021), Sep 10 2011 epoch, obtained with the MIRC beam combiner at H band.

There is no ground truth. We must extract the physics through careful modeling and cross-validation.

### 3.1 — Load and inspect the data


In [ ]:
lamand_file = "./data/2011Sep10_lam_And_prepped.oifits"
data_la = readoifits(lamand_file)[1,1]

println("λ And data loaded.")
println("  Baseline points: ", data_la.nuv)
println("  V² points:       ", data_la.nv2)
println("  T3 amplitudes:   ", data_la.nt3amp)
println("  Closure phases:  ", data_la.nt3phi)
println("  Max baseline:    ", round(maximum(data_la.uv_baseline), digits=1), " m")
res_la = mean(data_la.uv_lam) / (2*maximum(data_la.uv_baseline)) * 206265e3
println("  Resolution λ/2B: ", round(res_la, digits=3), " mas")


In [ ]:
uvplot(data_la)
println("Note the baseline distribution. Where are the gaps in UV coverage?")


In [ ]:
plot_v2(data_la; logplot=true, legend_below=true)
println("Where does V² first null? That baseline gives the approximate stellar diameter.")
println("For a uniform disk of diameter θ: first null at B = 1.22 λ/θ")


In [ ]:
plot_t3phi(data_la)
println("")
println("Are the closure phases consistent with zero?")
println("Significantly non-zero T3φ → brightness asymmetry → spots or companion.")
println("This is model-independent evidence for surface structure.")


### 3.2 — Estimate stellar diameter

Before imaging, fit a limb-darkened disk to the V² data. This gives us the stellar diameter for the **support mask** and a smooth **prior image** for BSMEM.

In [ ]:
# ── Quick diameter fit (V² only) ─────────────────────────────────────────────
# Fit a power-law limb-darkened disk to get the stellar diameter.
# This is just for the support mask and prior — 3 lines, not a fitting tutorial.

param_dict_la = Dict{String,Any}(
    "disk,ldpow" => 2.8,     # initial diameter guess (mas)
    "disk,alpha" => 0.25,    # LD power-law coefficient
    "disk,incl"  => 20.0,    # inclination (deg)
    "disk,pa"    => 42.0,    # position angle (deg)
    "disk,f"     => 1.0,
)
fit_params_la = ["disk,ldpow", "disk,alpha", "disk,incl", "disk,pa"]

result_la = fit_model(param_dict_la, fit_params_la, data_la;
    lb = Dict("disk,ldpow" => 1.5, "disk,alpha" => 0.0, "disk,incl" => 0.0, "disk,pa" => 0.0),
    ub = Dict("disk,ldpow" => 5.0, "disk,alpha" => 1.0, "disk,incl" => 60.0, "disk,pa" => 180.0),
    weights = [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],   # V² only
    method = :LN_NELDERMEAD)
println(result_la)

# Update param_dict with best-fit values
for (i, k) in enumerate(fit_params_la)
    param_dict_la[k] = result_la.x_opt[i]
end

In [ ]:
# Check the fit — does a smooth disk explain the V² data?
model_la = parse_model(param_dict_la, fit_params_la)
x_la_fit = Float64[param_dict_la[k] for k in fit_params_la]
obs_la = model_to_obs(model_la, x_la_fit, data_la)
plot_v2_residuals(data_la, obs_la.v2; logplot=true)
println("Systematic residuals → the data demand departures from a smooth disk (spots).")

### 3.3 — Support constraint and prior image

The **support** is the set of pixels where we allow non-zero flux. Restricting the reconstruction to the stellar disk:
1. Eliminates spurious flux at large separations (a very common artifact)
2. Adds prior knowledge without strongly biasing morphology

We use the fitted diameter to build a circular mask, and the fitted smooth disk image as the BSMEM prior.

In [ ]:
pixsize_la = 0.15   # mas/pixel
nx_la      = 64     # FOV = 9.6 mas

ft_la = setup_nfft(data_la, nx_la, pixsize_la)
println("Grid: $(nx_la)×$(nx_la) pixels, $(pixsize_la) mas/pixel")
println("FOV: $(nx_la * pixsize_la) mas")

In [ ]:
# Build circular support mask from fitted diameter
diam_mas = param_dict_la["disk,ldpow"]   # diameter in mas
major_pix = diam_mas / pixsize_la * 1.1   # radius in pixels, +10%

mask_la = zeros(nx_la, nx_la)
cx, cy = nx_la÷2+1, nx_la÷2+1
r = major_pix / 2
for ii in 1:nx_la, jj in 1:nx_la
    if (jj-cx)^2 + (ii-cy)^2 <= r^2
        mask_la[ii,jj] = 1.0
    end
end

# Prior image: smooth disk model multiplied by mask
prior_la = model_to_image(model_la, x_la_fit; nx=nx_la, pixsize=pixsize_la) .* mask_la
prior_la = prior_la / sum(prior_la)
# gaussian_prior() already includes a 1e-8 floor; we just ensure positivity
prior_la = max.(prior_la, 1e-8)
prior_la = prior_la / sum(prior_la)
println("Mask and prior image created.")
println("Mask coverage: $(round(100*sum(mask_la)/nx_la^2, digits=1))% of pixels")

In [ ]:
imdisp(mask_la; pixsize=pixsize_la, figtitle="Support mask")
imdisp(prior_la; pixsize=pixsize_la, figtitle="Prior image (smooth disk)")

### 3.4 — Reconstruction with gradient descent (TV)

First pass: TV regularization with the support constraint. Starting from the smooth disk model rather than a Gaussian gives faster convergence and a physically motivated starting point.


In [ ]:
x_start_la = prior_la / sum(prior_la)
chi2_f(x_start_la, ft_la, data_la; verb=true)
println("\nStarting chi² (smooth disk model): how much structure do the data require?")
println("If this is already near 1.0, the disk is smooth and there may be no spots.")
println("If it is large, the data demand departures from a smooth disk.")


In [ ]:
regularizers_la = [
    ["centering", 1e2],
    ["tv",        1e3],
    ["support",   1.0, mask_la],
]
weights_la = [1.0, 1.0, 1.0]   # [V², T3amp, T3phi]

println("Running TV reconstruction (~30 sec)...")
x_la = reconstruct(x_start_la, data_la, ft_la;
                    regularizers=regularizers_la,
                    weights=weights_la,   # remove if OITOOLS errors here
                    verb=false, maxiter=500)
println("Done.")


In [ ]:
imdisp(recenter(x_la; mask = x_la .> maximum(x_la)/10);
       pixsize=pixsize_la, use_colorbar=true)
chi2_f(x_la, ft_la, data_la; verb=true)


In [ ]:
# Mandatory data-fit check
v2_la, t3phi_la, t3amp_la = image_to_obs(x_la, ft_la, data_la)
plot_v2_residuals(data_la, v2_la)
plot_t3phi_residuals(data_la, t3phi_la)
println("If the model doesn't trace the data, the image is wrong regardless of how it looks.")


### 3.5 — L-curve: choosing the regularization weight

$\mu$ is the most consequential parameter. The **L-curve** sweeps $\mu$ over a range and plots $\chi^2$ vs regularization penalty on a log-log scale. The optimal $\mu$ is at the **corner of the L**:

- **Bottom-right** (small $\mu$): low regularization, low chi² → overfitting noise
- **Top-left** (large $\mu$): high regularization, high chi² → over-smoothed, real structure lost
- **The corner** → best balance

This sweep runs 5 reconstructions (~2.5 min total). Watch the chi²/N values as they print.


In [ ]:
tv_weights_la = [1e2, 5e2, 1e3, 5e3, 1e4]
lcurve_chi2_la = zeros(length(tv_weights_la))
lcurve_reg_la  = zeros(length(tv_weights_la))

for (i, mu) in enumerate(tv_weights_la)
    regs = [["centering", 1e2], ["tv", mu], ["support", 1.0, mask_la]]
    xl   = reconstruct(x_start_la, data_la, ft_la;
                       regularizers=regs, weights=weights_la, verb=false, maxiter=400)
    lcurve_chi2_la[i] = chi2_f(xl, ft_la, data_la)
    lcurve_reg_la[i]  = sum(abs.(diff(xl, dims=1))) + sum(abs.(diff(xl, dims=2)))
    println("μ = $mu  →  chi²/N = $(round(lcurve_chi2_la[i], digits=3))")
end
println("\nL-curve sweep complete.")

In [ ]:
figure(figsize=(7, 5))
loglog(lcurve_chi2_la, lcurve_reg_la .* tv_weights_la, "o-"; color="steelblue", markersize=8)
for (i, mu) in enumerate(tv_weights_la)
    annotate("μ=$(Int(mu))",
             xy=(lcurve_chi2_la[i], lcurve_reg_la[i]*tv_weights_la[i]),
             xytext=(6, 4), textcoords="offset points", fontsize=8)
end
xlabel("χ²/N  (data fidelity)")
ylabel("R(x)  (regularization penalty)")
title("L-curve: TV weight sweep — λ And Sep 10")
tight_layout()
gcf()


In [ ]:
# ── YOUR TURN ────────────────────────────────────────────────────────────────
# Pick the corner value from the L-curve above and run the final reconstruction.
# Try one value that's too small and one that's too large to see the effect.

mu_best_la = 1e3   # ← replace with your chosen value from the L-curve

regs_final_la = [
    ["centering", 1e2],
    ["tv",        mu_best_la],
    ["support",   1.0, mask_la],
]

println("Final reconstruction with μ = $mu_best_la ...")
x_la_final = reconstruct(x_start_la, data_la, ft_la;
                          regularizers=regs_final_la,
                          weights=weights_la,   # remove if OITOOLS errors here
                          verb=false, maxiter=500)

imdisp(recenter(x_la_final; mask = x_la_final .> maximum(x_la_final)/10);
       pixsize=pixsize_la, use_colorbar=true)
chi2_f(x_la_final, ft_la, data_la; verb=true)


### 3.6 — Null hypothesis: what does UV coverage artifact look like?

Before claiming a spot is real, ask: **could reconstruction artifacts from incomplete UV coverage produce the same structure?**

We answer this by reconstructing a *known* smooth model — a face-on limb-darkened disk with the same diameter and LD coefficient as λ And — using the exact same UV coverage as the real data. If this null hypothesis reconstruction looks smooth, any extra structure in the real reconstruction is genuinely demanded by the data.

This is the interferometric equivalent of asking: "what would I see if there were no spot?" 

In [ ]:
# Build a face-on LDD model using Martinez et al. (2021) Table 4 parameters
# incl=0 (face-on) removes any geometric asymmetry — pure null hypothesis
param_null = Dict{String,Any}(
    "disk,ldpow" => 2.742,   # angular diameter (mas) — Martinez et al.
    "disk,alpha" => 0.231,   # LD power-law coefficient — Martinez et al.
    "disk,incl"  => 0.0,     # face-on: no geometric elongation
    "disk,pa"    => 0.0,
    "disk,f"     => 1.0,
)
fit_params_null = ["disk,ldpow", "disk,alpha", "disk,incl", "disk,pa"]
model_null = parse_model(param_null, fit_params_null)
x_null_fit = Float64[param_null[k] for k in fit_params_null]

# Show the model image — this is what we are trying to reconstruct
x_ldd_img = model_to_image(model_null, x_null_fit; nx=nx_la, pixsize=pixsize_la)
imdisp(x_ldd_img; pixsize=pixsize_la, use_colorbar=true,
    figtitle="Null hypothesis model: face-on LDD")
println("Model: diameter=$(param_null[\"disk,ldpow\"]) mas, LD α=$(param_null[\"disk,alpha\"])")

In [ ]:
# Generate model observables at the exact UV points of the real data
obs_null = model_to_obs(model_null, x_null_fit, data_la)

# Inject into a copy of the real data structure — keep real error bars
data_null = deepcopy(data_la)
data_null.v2    .= obs_null.v2
data_null.t3phi .= obs_null.t3phi
data_null.t3amp .= obs_null.t3amp

println("Null hypothesis data created.")
println("Model V² range: $(round(minimum(obs_null.v2), digits=4)) — $(round(maximum(obs_null.v2), digits=4))")
println("Real V²  range: $(round(minimum(data_la.v2),  digits=4)) — $(round(maximum(data_la.v2),  digits=4))")

In [ ]:
# Reconstruct the null hypothesis data with the same settings as the real reconstruction
ft_null = setup_nfft(data_null, nx_la, pixsize_la)

println("Reconstructing null hypothesis (~30 sec)...")
x_null = reconstruct(prior_la / sum(prior_la), data_null, ft_null;
    regularizers = [
        ["centering", 1e2],
        ["tv",        1e3],
        ["support",   1.0, mask_la],
    ],
    maxiter = 300, verb = false)
println("Done.")

imdisp(recenter(x_null; mask = x_null .> maximum(x_null)/10);
    pixsize=pixsize_la, use_colorbar=true,
    figtitle="Null hypothesis reconstruction: smooth LDD, real UV coverage")

In [ ]:
# Side-by-side comparison: null hypothesis vs real reconstruction
x_null_c = recenter(x_null; mask = x_null .> maximum(x_null)/10)
x_real_c = recenter(x_la_final; mask = x_la_final .> maximum(x_la_final)/10)

cube_null = zeros(nx_la, nx_la, 2)
cube_null[:, :, 1] = x_null_c
cube_null[:, :, 2] = x_real_c

imdisp_multi(cube_null;
    labels=["Null hypothesis (smooth LDD)", "Real reconstruction (λ And)"],
    pixsize=pixsize_la, use_colorbar=true,
    figtitle="Null hypothesis test")

**Interpreting the null hypothesis test:**

- The left panel shows what reconstruction *artifacts* from this UV coverage look like — this is the worst case for a smooth star.
- The right panel is the real reconstruction.
- **Any structure in the right panel that is absent in the left panel is genuinely required by the data** — not a UV coverage artifact.
- If the two panels look similar, you cannot claim a spot detection with this dataset.
- If the right panel shows clear asymmetry absent from the left, you have evidence for real surface structure.

This test does **not** replace bootstrap validation, but it is fast and should always be done first.

### 3.6 — BSMEM reconstruction of λ And

Now run BSMEM on the same data with the same prior. BSMEM will only deviate from the smooth disk where the data strictly require it. Compare the spot locations between TV and BSMEM.


In [ ]:
println("Running BSMEM on λ And (~1 min)...")
x_la_bsmem = reconstruct_bsmem(prior_la, data_la, ft_la;
                 regularizers = [["mem", prior_la]],
                 method       = [4, 1, 1, 2],
                 maxiter      = 100,
                 verbose      = true,
                 flux_err     = 1e-5,
                 nrand        = 5)
println("Done.")

In [ ]:
x_la_c = recenter(x_la_final; mask = x_la_final .> maximum(x_la_final)/10)

cube_la = zeros(nx_la, nx_la, 2)
cube_la[:, :, 1] = x_la_c
cube_la[:, :, 2] = x_la_bsmem
c2_tv_la = round(chi2_f(x_la_final, ft_la, data_la), digits=3)
c2_bs_la = round(chi2_f(x_la_bsmem, ft_la, data_la), digits=3)
imdisp_multi(cube_la;
    labels=["TV μ=$(Int(mu_best_la)) (χ²/N=$c2_tv_la)", "BSMEM (χ²/N=$c2_bs_la)"],
    pixsize=pixsize_la, use_colorbar=true,
    figtitle="λ And — TV vs BSMEM")

**Interpreting the λ And reconstructions:**

1. **Spot locations**: identify any brightness asymmetries. Do the same regions appear in both TV and BSMEM? Features that appear in only one method are likely artifacts.
2. **Spot latitude**: is the dark region near the pole or at mid-latitude? Compare to the closure phases you inspected earlier.
3. **Resolution limit**: $\lambda/2B_{\max} \approx 0.5$ mas. Features smaller than this cannot be trusted, regardless of how they look.
4. **Limb darkening**: does the reconstructed disk show the expected brightness falloff at the limb, or is there unexpected structure there?


**What to look for:**

- With single-night data (45 V² per channel), each spectral channel is underconstrained — the reconstruction may collapse toward a smooth disk. This is expected.
- With multi-epoch combined data (6 nights × 45 = 270 V² per channel), structure becomes visible across channels.
- A real cool starspot has higher contrast at shorter wavelengths — look for the dark region to be darker at 1.5 μm than at 1.8 μm.
- A UV coverage artifact has no wavelength dependence — it appears identically in all channels.

**Key point:** polychromatic reconstruction requires significantly more data per channel than monochromatic. With MIRC-X/MYSTIC multi-epoch data this is a powerful spot reality check. With single-night data, use it to verify the disk diameter is consistent across wavelengths rather than to detect spots.

In [ ]:
# Load data with polychromatic splitting — one bin per spectral channel
data_poly = vec(readoifits(lamand_file; polychromatic=true))
println("Spectral channels: ", length(data_poly))
println("V² per channel:    ", data_poly[1].nv2)
println("Wavelength range:  ",
    round(minimum([mean(d.uv_lam) for d in data_poly])*1e6, digits=3), " — ",
    round(maximum([mean(d.uv_lam) for d in data_poly])*1e6, digits=3), " μm")

In [ ]:
# Set up polychromatic NFFT plan
ft_poly = setup_nfft_polychromatic(data_poly, nx_la, pixsize_la)

# Starting image cube: nx × nx × nchannels, same prior for each channel
nchannels = length(data_poly)
x0_poly = zeros(nx_la, nx_la, nchannels)
for i in 1:nchannels
    x0_poly[:, :, i] = prior_la / sum(prior_la)
end
println("Image cube size: $(nx_la)×$(nx_la)×$(nchannels)")

In [ ]:
# Set up polychromatic NFFT plan
ft_poly = setup_nfft_polychromatic(data_poly, nx_la, pixsize_la)

# Starting image cube: nx × nx × nchannels, same prior for each channel
nchannels = length(data_poly)
x0_poly = zeros(nx_la, nx_la, nchannels)
for i in 1:nchannels
    x0_poly[:, :, i] = prior_la / sum(prior_la)
end

# NOTE: regularizers must be a vector-of-vectors — one regularizer list per channel
# Support constraint is not available in polychromatic mode
regs_poly = [[ ["centering", 1e2], ["tv", 1e3] ] for _ in 1:nchannels]

println("Running polychromatic reconstruction (~2-3 min)...")
x_poly = reconstruct_polychromatic(x0_poly, data_poly, ft_poly;
    regularizers = regs_poly,
    maxiter = 200, verb = false)
println("Done. Image cube size: ", size(x_poly))

In [ ]:
# Show every 4th channel to span the H band
channel_step = 4
indices = collect(1:channel_step:nchannels)
n_show = length(indices)
wavs = [round(mean(data_poly[i].uv_lam)*1e6, digits=3) for i in indices]

cube_poly = zeros(nx_la, nx_la, n_show)
for (j, i) in enumerate(indices)
    xi = x_poly[:, :, i]
    cube_poly[:, :, j] = recenter(xi; mask = xi .> maximum(xi)/10)
end

imdisp_multi(cube_poly;
    labels=["$(w) μm" for w in wavs],
    pixsize=pixsize_la, use_colorbar=true,
    figtitle="Polychromatic reconstruction — λ And H band")

**What to look for:**

- Does the dark region (spot) appear consistently across all channels?
- Does its contrast increase toward shorter wavelengths (bluer = higher contrast for cool spots)?
- Any feature that appears in only one or two channels is likely noise or a calibration artifact.

A spot with temperature ~500 K below the photosphere (~4800 K) will have roughly **2× higher contrast** at 1.5 μm vs 1.8 μm. If you see this trend, the feature is almost certainly real.

---
## Part 4 — How Much Can You Trust the Reconstruction?

Image reconstruction always produces a result — even when the data are insufficient to constrain it. You must actively test whether features are real. Here are the three checks that should appear in every paper.

### 4.1 — Rule #1: The data-fit test

**If the reconstruction doesn't fit the data, the image is wrong.** This is non-negotiable.

Look at:
- V² residuals: systematic trends with baseline length → stellar size or shape is wrong
- T3φ residuals: large residuals on long baselines → missing fine-scale structure


In [ ]:
v2_f, t3p_f, t3a_f = image_to_obs(x_la_final, ft_la, data_la)

# These functions create their own figures
plot_v2_residuals(data_la, v2_f; logplot=true)
plot_t3phi_residuals(data_la, t3p_f)
println("λ And Sep 10 — data fit check complete")

### 4.2 — Rule #2: Method consistency

A feature that appears in **both** TV and BSMEM, with comparable chi²/N, is far more credible than one that appears in only one method.

Go back to the side-by-side panel above (Section 3.6) and systematically:
- Identify any dark regions (spots) in the TV image
- Check whether the same regions appear in the BSMEM image
- Are they at the same location and approximate contrast?

**If a spot appears in TV but not BSMEM:** it may be a TV staircase artifact — be very cautious.  
**If a spot appears in BSMEM but not TV:** it may be an over-smoothed prior pulling flux — also be cautious.  
**If a spot appears in both at similar location and contrast:** this is your most defensible claim.

### 4.3 — Rule #3: Bootstrap reliability

The standard reliability test is the **bootstrap**: resample the data with replacement, reconstruct 30–50 times, and compute the variance across images. A feature is "robust" if it appears in ≥80% of bootstrap realizations at ≥3σ contrast.

This is computationally expensive (30 × 30 sec = 15 min). The skeleton below is for use on your own machine after the workshop.


In [ ]:
#= BOOTSTRAP SKELETON — run at home (too slow for workshop)
n_boot = 30
x_boot = zeros(nx_la, nx_la, n_boot)

for b in 1:n_boot
    println("Bootstrap replicate $b / $n_boot")
    d_boot  = resample_data(data_la)
    ft_boot = setup_nfft(d_boot, nx_la, pixsize_la)
    regs_b  = [["centering", 1e2], ["tv", mu_best_la], ["support", 1.0, mask_la]]
    xb = reconstruct(prior_la / sum(prior_la), d_boot, ft_boot;
                     regularizers=regs_b, weights=[1.0,1.0,1.0],
                     verb=false, maxiter=400)
    x_boot[:, :, b] = xb
end

x_boot_mean = mean(x_boot; dims=3)[:, :, 1]
x_boot_std  = std(x_boot; dims=3)[:, :, 1]
x_boot_snr  = x_boot_mean ./ x_boot_std

imdisp(x_boot_snr; pixsize=pixsize_la,
       use_colorbar=true, figtitle="Bootstrap SNR map")
=#

println("Bootstrap skeleton ready — uncomment and run outside the workshop.")
println("Rule of thumb: a spot is robust if SNR > 3 in > 80% of bootstrap realisations.")

### 4.4 — Fundamental limits no amount of computation can overcome

| Limitation | Effect | Mitigation |
|---|---|---|
| Sparse UV coverage | Features at unsampled spatial frequencies are unconstrained | More baselines, longer tracks |
| Phase noise | Smearing of fine structure | Higher SNR, redundant closure phases |
| Calibration errors | Systematic bias in V² | Careful calibrator selection |
| Bandwidth smearing | Radial blurring in UV plane | Narrow spectral channels |
| Limited baseline range | No spatial frequencies below $B_{\min}$ or above $B_{\max}$ | Cannot recover extended or fine structure respectively |

The **resolution limit** $\lambda/2B_{\max}$ is absolute. Structure smaller than this cannot be imaged regardless of regularizer or algorithm.


---
## Part 5 — Bring Your Own Data

Use this template to apply the full workflow to your own OIFITS file. Fill in the values marked `# ← SET THIS` and run sequentially.

For instruments represented in this workshop: MIRCX (H band, $\lambda \approx 1.65$ μm), MYSTIC (K band, $\lambda \approx 2.2$ μm), PIONIER (H band), GRAVITY (K band), SPICA (visible).


In [ ]:
# ── YOUR TURN: Bring Your Own Data ───────────────────────────────────────────

# 1. Path to your OIFITS file (upload to Binder via the file browser, or use a URL)
my_oifits = "./data/your_file.oifits"   # ← SET THIS

# 2. Load
my_data = readoifits(my_oifits)[1,1]    # adjust epoch/wavelength indices if multi-epoch

# 3. Inspect
println("Your data:")
println("  V² points:      ", my_data.nv2)
println("  Closure phases: ", my_data.nt3phi)
println("  Max baseline:   ", round(maximum(my_data.uv_baseline), digits=1), " m")
res_my = mean(my_data.uv_lam) / (2*maximum(my_data.uv_baseline)) * 206265e3
println("  Resolution λ/2B: ", round(res_my, digits=3), " mas")


In [ ]:
# 4. Visualise UV coverage and observables
uvplot(my_data)


In [ ]:
plot_v2(my_data; logplot=true, legend_below=true)
plot_t3phi(my_data)


In [ ]:
# 5. Set up the image grid
#    pixsize: start at λ/(2Bmax) or half that for safety
#    nx:      choose so FOV = nx × pixsize is ~3–5× your source diameter estimate

my_pixsize = res_my / 2.0   # ← refine this
my_nx      = 64             # ← refine this; must be a power of 2 or factor of small primes

my_ft = setup_nfft(my_data, my_nx, my_pixsize)
println("Grid: $(my_nx)×$(my_nx), pixsize=$(round(my_pixsize,digits=3)) mas/pixel")
println("FOV: $(round(my_nx*my_pixsize, digits=2)) mas")


In [ ]:
# 6. Quick diameter estimate for support mask
my_param_dict = Dict{String,Any}("star,ud" => 1.0, "star,f" => 1.0)
my_result = fit_model(my_param_dict, ["star,ud"], my_data;
    lb = Dict("star,ud" => 0.1), ub = Dict("star,ud" => 10.0),
    weights = [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],   # V² only
    method = :LN_NELDERMEAD)
println(my_result)
my_param_dict["star,ud"] = my_result.x_opt[1]

In [ ]:
# 7. Build support mask and prior
diam_pix  = my_param_dict["star,ud"] / my_pixsize * 1.1   # inflate by 10%
my_mask = zeros(my_nx, my_nx)
cx_m, cy_m = my_nx÷2+1, my_nx÷2+1
r = diam_pix/2
for i in 1:my_nx, j in 1:my_nx
    if (j-cx_m)^2 + (i-cy_m)^2 <= r^2
        my_mask[i,j] = 1.0
    end
end
my_prior  = gaussian2d(my_nx, my_nx, diam_pix/3)
my_prior  = my_prior .* my_mask
my_prior /= sum(my_prior)

imdisp(my_mask; pixsize=my_pixsize)

In [ ]:
# 8. TV reconstruction
my_regs = [
    ["centering", 1e2],
    ["tv",        1e3],   # ← tune this with an L-curve (see Part 3.5)
    ["support",   1.0, my_mask],
]

println("Running TV reconstruction...")
x_my = reconstruct(my_prior, my_data, my_ft;
                    regularizers=my_regs,
                    weights=[1.0, 1.0, 1.0],   # remove if OITOOLS errors here
                    verb=false, maxiter=500)

imdisp(recenter(x_my; mask = x_my .> maximum(x_my)/10);
       pixsize=my_pixsize, use_colorbar=true)
chi2_f(x_my, my_ft, my_data; verb=true)

In [ ]:
# 9. Data-fit check — mandatory
v2_my, t3p_my, t3a_my = image_to_obs(x_my, my_ft, my_data)
plot_v2_residuals(my_data, v2_my)
plot_t3phi_residuals(my_data, t3p_my)


In [ ]:
# 10. BSMEM for cross-validation
println("Running BSMEM...")
x_my_bsmem = reconstruct_bsmem(my_prior, my_data, my_ft;
                 regularizers = [["mem", my_prior]],
                 method       = [4, 1, 1, 2],
                 maxiter      = 100,
                 verbose      = false,
                 flux_err     = 1e-5,
                 nrand        = 5)

x_my_c = recenter(x_my; mask = x_my .> maximum(x_my)/10)
cube_my = zeros(my_nx, my_nx, 2)
cube_my[:, :, 1] = x_my_c
cube_my[:, :, 2] = x_my_bsmem
imdisp_multi(cube_my;
    labels=["TV reconstruction", "BSMEM cross-check"],
    pixsize=my_pixsize, use_colorbar=true)

---
*Workshop data: Martinez et al. (2021) ApJ — λ And CHARA/MIRC H-band*  
*OITOOLS.jl: Baron (2022) — https://github.com/fabienbaron/OITOOLS.jl*  
*Notebook: Ryan Norris, Fabien Baron — New Visions in Optical Interferometry, March 2026* with help from Claude AI
